# Chit 5 baseline: a mosquito detector for a bed-net sensor · **MCU**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

A battery-powered sensor by a bed net listens all night and counts mosquitoes, so that only tiny nightly summaries leave the device. Train a small CNN on the paper's log-mel features of 1.92-s audio windows, measure **mosquito recall at ≤ 1% false alarms** and PR-AUC on **unseen sites** (Tanzania field, UK lab cultures) in fp32 and int8, and count the cost **including the feature extraction**.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~3 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~1 min) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["onnx", "onnxruntime", "soundfile"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 0. Understand the data
Before any model, look at the data: what one sample is, what the classes look like, how the splits were made, and why the chit uses its metric.
This part only reads and plots (≲ 1 min). The model starts in Part 1.

### 0.1 The task in plain words
Malaria is spread by *Anopheles* mosquitoes. Health teams need to know where and when mosquitoes are active, to decide where to spray or hand out bed nets.
A flying mosquito makes a high-pitched hum: its wings beat several hundred times per second. A cheap sensor next to a bed net can listen all night, detect that hum on the device,
and send only a short summary each morning (how many detections, and when).
**One decision of the model** = for one 1.92-second window of sound, "a mosquito is flying here" or "no mosquito".

### 0.2 What one sample is
One sample is a **window of raw audio**: 15,360 signed 16-bit integers = **1.92 s at 8,000 samples per second**. Each number is the microphone signal at one instant;
divide by 32,768 to get a fraction of the recorder's full scale (1 = the loudest the recorder can store). There is no calibration to physical units, and the experiments used
different phones and microphones, so loudness is only comparable within one experiment.
**Label:** 1 = mosquito, 0 = no mosquito ("background": silence, voices, other insects, machines). HumBugDB experts labelled whole clips; every window of a clip gets the clip's label.
`meta()` tells you where each window comes from: the **experiment** (country / setting / place), and for test windows, test set A or B.
Below: two simple numbers per window, the loudness (RMS, in dB) and the frequency with the most energy, for a sample of training windows and for all test windows.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from capstone.data.humbug import load, meta, logmel, mel_filterbank, WINDOW, SR
A0, y0 = load("train", max_items=6_000); M0 = meta("train", max_items=6_000)      # a representative prefix of the training windows
A1, y1 = load("test"); M1 = meta("test")
print("one window:", A0.shape[1:], A0.dtype, f"= {WINDOW / SR:.2f} s at {SR} samples/s | label 1 = mosquito, 0 = no mosquito")
print("first training window: label", y0[0], "| experiment:", M0["experiment"][0], "| first samples:", A0[0, :6])
def describe(A):
    x = A.astype(np.float32) / 32768; x = x - x.mean(1, keepdims=True)
    rms_db = 10 * np.log10((x ** 2).mean(1) + 1e-12)                          # loudness in dB relative to full scale (0 dB = the loudest possible)
    P = np.abs(np.fft.rfft(x * np.hanning(x.shape[1]), axis=1)) ** 2; f = np.fft.rfftfreq(x.shape[1], 1 / SR)
    return rms_db, f[1:][np.argmax(P[:, 1:], 1)], P[:, f < 100].sum(1) / P.sum(1)
(rms0, peak0, low0), (rms1, peak1, low1) = describe(A0), describe(A1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for (rms, peak, y, where, ls) in ((rms0, peak0, y0, "train", "-"), (rms1, peak1, y1, "test", "--")):
    for k, c, name in ((1, "C3", "mosquito"), (0, "C0", "no mosquito")):
        ax[0].hist(rms[y == k], np.linspace(-90, 0, 61), density=True, histtype="step", color=c, ls=ls, label=f"{name}, {where}")
        ax[1].hist(peak[y == k], np.logspace(0, np.log10(4000), 61), weights=np.full((y == k).sum(), 1 / (y == k).sum()), histtype="step", color=c, ls=ls,
                   label=f"{name}, {where}")
ax[0].set(xlabel="loudness of the window (RMS) [dB relative to full scale]", ylabel="density", title="Loudness")
ax[1].set(xlabel="frequency with the most energy [Hz]", ylabel="fraction of windows", xscale="log", yscale="log", title="Frequency with the most energy (grey: 300-1000 Hz)")
ax[1].axvspan(300, 1000, color="0.9", zorder=0)
for a in ax:
    a.legend(fontsize=7)
plt.tight_layout(); plt.show()
for name, low, y in (("train", low0, y0), ("test", low1, y1)):
    print(f"{name}: median share of the energy below 100 Hz: mosquito {np.median(low[y == 1]):.0%}, no mosquito {np.median(low[y == 0]):.0%}")
for name, peak, y in (("train", peak0, y0), ("test", peak1, y1)):
    print(f"{name}: mosquito windows whose strongest frequency is below 30 Hz: {np.mean(peak[y == 1] < 30):.0%}, between 300 and 1000 Hz: "
          f"{np.mean((peak[y == 1] >= 300) & (peak[y == 1] <= 1000)):.0%}")

**What to notice.** Left: mosquito windows are not simply louder; loudness depends mostly on the recording (test windows are much quieter than training windows,
and the narrow peaks of the test curves come from a few long recordings, or from one experiment's recorder). Right: this is the surprise. In **training**, 83% of the mosquito windows have their strongest frequency below 30 Hz
(a very low, deep sound, not the hum) and almost all their energy is below 100 Hz, while the no-mosquito windows have much less there. In **test**, the mosquito windows look as expected:
60% have their strongest frequency between 300 and 1000 Hz (the wingbeat), and almost no energy below 100 Hz.
So in training, "lots of energy below 100 Hz" goes with "mosquito", probably because of how the training recordings were made: almost all training mosquito windows come from mosquitoes recorded in cups (see the table in 0.4).
That clue does not exist in the field. A model that uses it will look good on `val` and fail on `test`. Think about which frequencies your features should keep.

### 0.3 Look at it (and listen)
Four random windows: a mosquito and a no-mosquito window from training, then from test A (the field recordings in Tanzania).
Top: the waveform. Bottom: the log-mel spectrogram the baseline uses (128 frequency bands on the mel scale, which spreads the low frequencies; 30 time frames).
Then listen: the players below play the same four windows.

In [ ]:
rng = np.random.default_rng(0)
cases = [("train", "mosquito", A0, rng.choice(np.where(y0 == 1)[0]), M0), ("train", "no mosquito", A0, rng.choice(np.where(y0 == 0)[0]), M0),
         ("test A", "mosquito", A1, rng.choice(np.where((y1 == 1) & (M1["test_set"] == "A"))[0]), M1),
         ("test A", "no mosquito", A1, rng.choice(np.where((y1 == 0) & (M1["test_set"] == "A"))[0]), M1)]
fb = mel_filterbank(); band_hz = fb.argmax(1) * SR / 2048              # the centre frequency of each of the 128 mel bands
fig, ax = plt.subplots(2, 4, figsize=(17, 6.5))
for j, (where, name, A, i, M) in enumerate(cases):
    ax[0, j].plot(np.arange(WINDOW) / SR, A[i] / 32768, lw=0.4, color="C3" if name == "mosquito" else "C0")
    ax[0, j].set(xlabel="time [s]", ylabel="amplitude (1 = full scale)", title=f"{where}, {name}\n{M['experiment'][i]}")
    im = ax[1, j].imshow(logmel(A[i:i + 1])[0, 0], aspect="auto", origin="lower", cmap="magma", extent=(0, WINDOW / SR, 0, 128))
    ticks = [100, 300, 500, 1000, 2000, 4000]
    ax[1, j].set_yticks([np.argmin(np.abs(band_hz - t)) for t in ticks], [str(t) for t in ticks])
    ax[1, j].set(xlabel="time [s]", ylabel="frequency [Hz] (mel scale)", title="log-mel spectrogram (what the model sees)")
    plt.colorbar(im, ax=ax[1, j], label="standardised log power")
plt.tight_layout(); plt.show()
for where, name, A, i, M in cases:
    print(f"{where}, {name} ({M['experiment'][i]}):")
    display(Audio(A[i] / 32768, rate=SR))          # the player normalises the loudness: real windows are much quieter than this

**What to notice** (in these four windows; look at more). The training mosquito window has its brightest band at the very bottom (below 100 Hz), with
faint lines near 500 and 1000 Hz. The training no-mosquito window has stacked lines that change pitch: a voice or an animal call, loud and clear. In the field mosquito window,
the mosquito is short and faint: a tone near 500 Hz and its harmonic near 1000 Hz, only around 1.5 s, where the waveform also shows a burst. The field no-mosquito window has
steady high tones (about 2 and 4 kHz) that are not mosquitoes. Field recordings are quiet: the amplitudes are a few thousandths of full scale.
(The players normalise the loudness, so quiet windows sound louder than they are.)

### 0.4 The numbers that matter

In [ ]:
print("split    windows  hours  mosquito")
for name in ("train", "val", "test"):
    m = meta(name)
    for part, sel in [(name, np.ones(len(m["label"]), bool))] + ([("test A", m["test_set"] == "A"), ("test B", m["test_set"] == "B")] if name == "test" else []):
        print(f"{part:<7} {sel.sum():>8,} {sel.sum() * WINDOW / SR / 3600:6.1f}  {m['label'][sel].mean():7.1%}")
print("\nper experiment (country / setting / place):")
for name in ("train", "val", "test"):
    m = meta(name)
    for e in np.unique(m["experiment"]):
        sel = m["experiment"] == e
        print(f"  {name:<5} {e:<52} {sel.sum():>7,} windows  {m['label'][sel].mean():6.1%} mosquito")

**Splits.** `test` = the paper's two test sets, unchanged: **A** = field recordings in Tanzania, **B** = lab cultures in the UK (Oxford and London).
They are whole experiments that never appear in training. The rest is split **by recording** (a recording is never in two splits): within each experiment, recordings
are shuffled once (fixed seed) and moved to `val` until it holds 15% of that experiment's audio. So `val` comes from the **same experiments as train** (same rooms, same recorders):
that is why `val` is much too optimistic.
**Class balance per experiment.** Look at the table: Kenya cultures are 95% no-mosquito, the Tanzania cups 88% mosquito, the USA cultures 0% mosquito. Inside the training
data, knowing the experiment almost gives the label away, and so does the recording setup (the low-frequency rumble above). Neither helps on test.
**In the field** only about 1–2% of the windows contain a mosquito; here the test set is 42% mosquito.
**Value range:** int16 (−32,768 to 32,767); most windows use only a small part of it (RMS mostly between −35 and −75 dB of full scale).

### 0.5 The metric, explained on the data
A sensor listens for ~12 hours a night, about 22,500 windows, and in the field only 1–2% of them contain a mosquito. If even 5% of the empty windows raised a false alarm,
the false alarms would outnumber the real mosquitoes several times, and the nightly count would be useless. So the chit fixes the false-alarm rate (**≤ 1% of the no-mosquito
windows**) and asks how many mosquito windows are found at that threshold (recall). Accuracy would be misleading because it depends on the class mix, which is 42% here
and 1–2% in the field. PR-AUC is a second, threshold-free number; it also depends on the mix, so it is optimistic for the field.

In [ ]:
del A0, A1

## 1. Data
One sample = 1.92 s of audio at 8 kHz (15,360 int16 samples); label 1 = mosquito. `test` holds two whole experiments the model never sees in training:
**A** = field recordings in Tanzania, **B** = lab cultures in the UK (`meta("test")`). No recording is in two splits.

In [ ]:
from capstone.data.humbug import load, meta, logmel, logmel_macs, recall_at_false_alarm, pr_auc, WINDOW, SR
A_tr, y_tr = load("train", max_items=4_000 if FAST else None)
A_va, y_va = load("val"); A_te, y_te = load("test")
TEST_SET = meta("test")["test_set"]
for name, y in (("train", y_tr), ("val", y_va), ("test", y_te), ("test A", y_te[TEST_SET == "A"]), ("test B", y_te[TEST_SET == "B"])):
    print(f"{name:<6} {len(y):>7,} windows ({len(y) * WINDOW / SR / 3600:5.1f} h), {np.mean(y):.1%} mosquito")

## 2. Features: the paper's log-mel spectrogram, and what it costs
`logmel` turns each window into 128 mel bands × 30 frames (2048-point FFT, hop 512). On the device this runs for every window, so its
multiply-accumulates count toward the budget: `logmel_macs()`. FFT size, hop, number of bands and sample rate are all levers.

In [ ]:
t0 = time.time()
XTR, XVA, XTE = logmel(A_tr), logmel(A_va), logmel(A_te)
print(f"features {XTR.shape[1:]} for {len(XTR) + len(XVA) + len(XTE):,} windows in {time.time() - t0:.0f} s; {logmel_macs():,} MACs per window")
A_one = A_te[:1].copy()
del A_tr, A_va, A_te                              # the raw audio is no longer needed: free the memory

## 3. The model, costed before training
Three strided 3 × 3 convolutions (128 × 30 → 16 × 4), global average pooling, a linear layer. `extra_macs=logmel_macs()` adds the features.

In [ ]:
model = nn.Sequential(nn.Conv2d(1, 8, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                      nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 2))
p = profile(model, (1, 128, 30), extra_macs=logmel_macs())
print(f"params {p['params']:,} | MACs {p['macs']:,} (of which log-mel {logmel_macs():,}) | int8 weights {p['weight_bytes']:,} B | peak act {p['peak_act_bytes']:,} B")
WIN_S = WINDOW / SR
print(f"latency at 100 M MAC/s: {1e3 * mcu_latency(p['macs'], 100e6):.1f} ms per {WIN_S:.2f}-s window "
      f"(the device is busy {mcu_latency(p['macs'], 100e6) / WIN_S:.1%} of the time)")
E = energy_uj(p)
print(f"energy (card formula): {E:.2f} µJ per window = {E * 3600 / WIN_S / 1e3:.2f} mJ per hour of listening")
TARGET = dict(max_weight_bytes=16 * 1024, max_act_bytes=32 * 1024, mac_rate=50e6, max_latency_s=0.100)   # PROVISIONAL: your chit card has the budgets
fits(p, TARGET);

## 4. Train and measure
The threshold is set on the test set's no-mosquito windows so that 1% of them raise a false alarm; recall is measured at that threshold.

In [ ]:
fit(model, XTR, y_tr, epochs=1 if FAST else 10, bs=128, lr=3e-3)
def scores(out):
    return torch.softmax(torch.as_tensor(out), 1)[:, 1].numpy()
s32 = scores(predict(model, XTE))
REC32, AP32 = recall_at_false_alarm(s32, y_te), pr_auc(s32, y_te)
sva = scores(predict(model, XVA))
print(f"val (sites seen in training): recall @ 1% false alarms {recall_at_false_alarm(sva, y_va):.3f}, PR-AUC {pr_auc(sva, y_va):.3f}")
print(f"test (unseen sites), fp32:    recall @ 1% false alarms {REC32:.3f}, PR-AUC {AP32:.3f}")

## 5. int8 (weights and activations), with ONNX Runtime static quantization
An MCU runs int8: this is the number the chit counts.

In [ ]:
to_onnx(model.cpu(), XTR[:1], "my_models/humbug_fp32.onnx")
ort_int8("my_models/humbug_fp32.onnx", XTR[np.random.default_rng(0).permutation(len(XTR))[:512]], "my_models/humbug_int8.onnx")
sess8 = ort_session("my_models/humbug_int8.onnx")
OUT8 = ort_predict(sess8, XTE)

s8 = scores(OUT8)
REC8, AP8 = recall_at_false_alarm(s8, y_te), pr_auc(s8, y_te)
print(f"test, int8: recall @ 1% false alarms {REC8:.3f}, PR-AUC {AP8:.3f}   <- the chit's metric")
for ts in ("A", "B"):
    m = TEST_SET == ts
    print(f"  test {ts}: recall @ 1% false alarms {recall_at_false_alarm(s8[m], y_te[m]):.3f}, PR-AUC {pr_auc(s8[m], y_te[m]):.3f}")
print("other false-alarm rates (int8): " + ", ".join(f"{fa:.1%} -> recall {recall_at_false_alarm(s8, y_te, fa):.3f}" for fa in (0.05, 0.02, 0.005)))

**What 1% false alarms means in the field.** The test set is about half mosquito; a real night is ~1-2% mosquito. With prevalence π, recall r and
false-alarm rate f, the fraction of the device's detections that are real mosquitoes is r·π / (r·π + f·(1 − π)):

In [ ]:
for prev in (0.01, 0.02):
    print(f"at {prev:.0%} mosquito time: {REC8 * prev / (REC8 * prev + 0.01 * (1 - prev)):.0%} of the detections are real")

## 6. Sanity measurement (required for MCU chits): ONNX Runtime int8 on 1 thread
Far faster than a microcontroller; what matters is whether your candidates **rank** like their MAC counts. The log-mel is timed too (NumPy here):
it is half the MACs, but it is FFTs and a log, not a convolution.

In [ ]:
from common.timing import measure
LAT8 = ort_latency(sess8, XTE[:1])
LATF = measure(lambda: logmel(A_one))
print(f"CNN int8, ORT 1 thread: median {LAT8['median_us']:.0f} µs, p99 {LAT8['p99_us']:.0f} µs | log-mel (NumPy): median {LATF['median_us']:.0f} µs, p99 {LATF['p99_us']:.0f} µs")

In [ ]:
ROW = report("5 humbug", (len(XTR), len(XVA), len(XTE)), (1, 128, 30),
             f"recall@1%FA int8 {REC8:.3f} PR-AUC {AP8:.3f} (fp32 {REC32:.3f}/{AP32:.3f})", p,
             f"MCU counted: {1e3 * mcu_latency(p['macs'], 100e6):.1f} ms per 1.92-s window at 100 M MAC/s incl. log-mel; "
             f"ORT int8 1 thread p99 {LAT8['p99_us']:.0f} µs (CNN only); energy {E:.2f} µJ/window",
             "CC BY 4.0 (Zenodo, confirmed)", extra=dict(logmel_macs=logmel_macs(), energy_uj=round(E, 3), minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.